# 02 Build Gold Forecasting Features Table

This notebook constructs `gold_forecasting_features` — the ML-ready dataset for 1-day-ahead demand forecasting. All lag, rolling, and zero-demand features are **calendar-correct** via a complete daily date spine constructed per store.

## Architecture: Store-Batched DuckDB Execution
The base fact table contains **105.9M rows**. Running global window functions over all 54 stores simultaneously requires >16 GB of RAM and triggers allocation limits. By processing the dataset **store by store** (54 partitions of ~1.96M rows each), memory usage stays under 50 MB per store, and the entire 105.9M row dataset is featured and saved in ~3 minutes.

## Feature Mapping (28 Columns, No Target Leakage)
```
forecast_date D (Decision Time)
   ├── units_sold            = today's sales
   ├── lag_1                 = sales on D-1
   ├── lag_7                 = sales on D-7
   ├── lag_14                = sales on D-14
   ├── lag_28                = sales on D-28
   ├── rolling_mean_7, 14, 28 = mean of [D-7..D-1], [D-14..D-1], [D-28..D-1]
   ├── rolling_std_7, 28     = std of [D-7..D-1], [D-28..D-1]
   ├── zero_sales_count_7, 28 = count of zero-sales days in prior 7 & 28 days
   ├── family, is_perishable = static product metadata
   ├── city, store_type, cluster = static store metadata
   ├── target_day_of_week, target_month, target_is_weekend, target_is_holiday
   ├── on_promotion_today    = promo status on D (known now)
   ├── on_promotion_tomorrow = promo status on D+1 (known future)
   └── oil_price             = economic indicator on D (known now)
                 ↓
target_unit_sales            = sales on D+1   ← TARGET: never in features
```

In [1]:
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path
import time
import shutil

# Dynamic lakehouse root resolution
lakehouse_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "data_lake").is_dir() and (p / "notebooks").is_dir():
        lakehouse_root = p
        break

if lakehouse_root is None:
    lakehouse_root = Path("D:/Demand_forecasting/inventory-lakehouse")

GOLD_DIR      = lakehouse_root / "data_lake" / "gold"
BASE_PATH     = GOLD_DIR / "gold_forecasting_base.parquet"
FEATURES_DIR  = GOLD_DIR / "gold_forecasting_features"

print(f"Lakehouse root : {lakehouse_root}")
print(f"DuckDB version : {duckdb.__version__}")
print(f"Base path      : {BASE_PATH}")
print(f"Features dir   : {FEATURES_DIR}")

assert BASE_PATH.exists(), "gold_forecasting_base.parquet not found. Run Notebook 01 first."

## Step 1 — Validate Base Grain & Fetch Store List

In [2]:
print("Validating base grain and getting store list...")
t0 = time.time()
con = duckdb.connect()

base_str = str(BASE_PATH).replace("\\", "/")

# Validate grain uniqueness
dup_check = con.execute(f"""
    SELECT COUNT(*) 
    FROM (
        SELECT forecast_date, store_id, sku_id, COUNT(*) AS cnt
        FROM read_parquet('{base_str}')
        GROUP BY forecast_date, store_id, sku_id
        HAVING cnt > 1
    )
""").fetchone()[0]

assert dup_check == 0, f"STOP: {dup_check:,} duplicate grain keys found!"

# Fetch unique stores
stores = [r[0] for r in con.execute(f"""
    SELECT DISTINCT store_id 
    FROM read_parquet('{base_str}') 
    ORDER BY store_id
""").fetchall()]

print(f"[OK] Grain unique. Found {len(stores)} physical stores to process.")
print(f"     Validation elapsed: {time.time()-t0:.1f}s")

## Step 2 — Execute Store-Batched Date Spine & Feature Pipeline
For each store: build calendar date spine, left-join actual sales (missing = 0.0), compute 18 features (lags, rollings, zero-demand), join base metadata, filter `lag_28 IS NOT NULL`, and save store partition.

In [3]:
if FEATURES_DIR.exists():
    shutil.rmtree(FEATURES_DIR)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processing features for {len(stores)} stores...")
t0_all = time.time()
total_feature_rows = 0

for idx, sid in enumerate(stores, 1):
    t0 = time.time()
    out_part = FEATURES_DIR / f"store_id={sid}.parquet"
    out_part_str = str(out_part).replace("\\", "/")
    
    sql_store = f"""
        COPY (
            WITH store_base AS (
                SELECT * FROM read_parquet('{base_str}') WHERE store_id = {sid}
            ),
            pair_bounds AS (
                SELECT 
                    sku_id,
                    MIN(forecast_date) AS min_date,
                    MAX(forecast_date) AS max_date
                FROM store_base
                GROUP BY sku_id
            ),
            spine AS (
                SELECT 
                    p.sku_id,
                    CAST(d.forecast_date AS DATE) AS forecast_date
                FROM pair_bounds p,
                LATERAL UNNEST(generate_series(p.min_date, p.max_date, INTERVAL 1 DAY)) AS d(forecast_date)
            ),
            spine_sales AS (
                SELECT 
                    s.sku_id,
                    s.forecast_date,
                    {sid} AS store_id,
                    COALESCE(b.units_sold, 0.0) AS units_sold
                FROM spine s
                LEFT JOIN store_base b 
                    ON b.sku_id = s.sku_id AND b.forecast_date = s.forecast_date
            ),
            spine_features AS (
                SELECT 
                    forecast_date, store_id, sku_id, units_sold,
                    LAG(units_sold, 1)  OVER w AS lag_1,
                    LAG(units_sold, 7)  OVER w AS lag_7,
                    LAG(units_sold, 14) OVER w AS lag_14,
                    LAG(units_sold, 28) OVER w AS lag_28,
                    AVG(units_sold)    OVER (w ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)  AS rolling_mean_7,
                    AVG(units_sold)    OVER (w ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING) AS rolling_mean_14,
                    AVG(units_sold)    OVER (w ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_mean_28,
                    STDDEV_SAMP(units_sold) OVER (w ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)  AS rolling_std_7,
                    STDDEV_SAMP(units_sold) OVER (w ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_std_28,
                    SUM(CASE WHEN units_sold = 0 THEN 1 ELSE 0 END) OVER (w ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)  AS zero_sales_count_7d,
                    SUM(CASE WHEN units_sold = 0 THEN 1 ELSE 0 END) OVER (w ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS zero_sales_count_28d
                FROM spine_sales
                WINDOW w AS (PARTITION BY sku_id ORDER BY forecast_date)
            )
            SELECT 
                f.forecast_date,
                b.target_date,
                f.store_id,
                f.sku_id,
                f.units_sold,
                f.lag_1,
                f.lag_7,
                f.lag_14,
                f.lag_28,
                f.rolling_mean_7,
                f.rolling_mean_14,
                f.rolling_mean_28,
                f.rolling_std_7,
                f.rolling_std_28,
                f.zero_sales_count_7d,
                f.zero_sales_count_28d,
                b.family,
                b.is_perishable,
                b.city,
                b.store_type,
                b.cluster,
                b.target_day_of_week,
                b.target_month,
                b.target_is_weekend,
                b.target_is_holiday,
                b.on_promotion_today,
                b.on_promotion_tomorrow,
                b.oil_price,
                b.target_unit_sales
            FROM spine_features f
            INNER JOIN store_base b
                ON b.sku_id = f.sku_id AND b.forecast_date = f.forecast_date
            WHERE f.lag_28 IS NOT NULL
            ORDER BY f.forecast_date, f.sku_id
        ) TO '{out_part_str}' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """
    con.execute(sql_store)
    
    part_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_part_str}')").fetchone()[0]
    total_feature_rows += part_rows
    
    if idx % 10 == 0 or idx == len(stores):
        print(f"  - Processed {idx}/{len(stores)} stores. Total rows so far: {total_feature_rows:,} [{time.time()-t0_all:.1f}s]")

print(f"\n[OK] All {len(stores)} store partitions processed successfully!")
print(f"     Total feature rows: {total_feature_rows:,}")
print(f"     Total pipeline elapsed: {time.time()-t0_all:.1f}s")

## Step 3 — Automated Lag Validation & Visual Spot-Check

In [4]:
feat_glob_str = str(FEATURES_DIR / "*.parquet").replace("\\", "/")

print("Running automated lag correctness assertions...")

# Automated check on 3 complete store partitions
val_sql = f"""
    WITH val_slice AS (
        SELECT forecast_date, store_id, sku_id, units_sold, lag_1, lag_7
        FROM read_parquet('{feat_glob_str}')
        WHERE store_id IN (1, 2, 3)
    ),
    check_lag1 AS (
        SELECT 
            v.forecast_date, v.store_id, v.sku_id, v.lag_1, prev.units_sold AS expected_lag1
        FROM val_slice v
        JOIN val_slice prev
            ON prev.store_id = v.store_id 
           AND prev.sku_id = v.sku_id 
           AND prev.forecast_date = v.forecast_date - INTERVAL 1 DAY
    ),
    check_lag7 AS (
        SELECT 
            v.forecast_date, v.store_id, v.sku_id, v.lag_7, prev.units_sold AS expected_lag7
        FROM val_slice v
        JOIN val_slice prev
            ON prev.store_id = v.store_id 
           AND prev.sku_id = v.sku_id 
           AND prev.forecast_date = v.forecast_date - INTERVAL 7 DAY
    )
    SELECT 
        (SELECT COUNT(*) FROM check_lag1 WHERE lag_1 != expected_lag1) AS mismatches_lag1,
        (SELECT COUNT(*) FROM check_lag7 WHERE lag_7 != expected_lag7) AS mismatches_lag7,
        (SELECT COUNT(*) FROM check_lag1) AS eval_rows_lag1,
        (SELECT COUNT(*) FROM check_lag7) AS eval_rows_lag7
"""

val_res = con.execute(val_sql).fetchone()
mismatches_lag1, mismatches_lag7, eval_l1, eval_l7 = val_res

assert mismatches_lag1 == 0, f"ERR: lag_1 validation failed: {mismatches_lag1} mismatches!"
print(f"[OK] Automated Lag-1 Assertion Passed: lag_1(D) == units_sold(D-1) across {eval_l1:,} evaluated rows.")

assert mismatches_lag7 == 0, f"ERR: lag_7 validation failed: {mismatches_lag7} mismatches!"
print(f"[OK] Automated Lag-7 Assertion Passed: lag_7(D) == units_sold(D-7) across {eval_l7:,} evaluated rows.")

# Visual spot check sample
sample_df = con.execute(f"""
    SELECT 
        forecast_date, units_sold, lag_1, lag_7, rolling_mean_7, zero_sales_count_7d, target_unit_sales
    FROM read_parquet('{feat_glob_str}')
    WHERE store_id = 1 AND sku_id = 105574
    ORDER BY forecast_date
    LIMIT 10
""").df()

print("\nVisual Spot-Check -- Store 1, SKU 105574 (10 consecutive rows):")
print(sample_df)

# Final Dataset Metadata Summary
summary = con.execute(f"""
    SELECT 
        COUNT(*) AS total_rows,
        MIN(forecast_date) AS min_date,
        MAX(forecast_date) AS max_date,
        COUNT(DISTINCT store_id) AS stores,
        COUNT(DISTINCT sku_id) AS skus
    FROM read_parquet('{feat_glob_str}')
""").fetchone()

print("\n=== FINAL FEATURE DATASET SUMMARY ===")
print(f"   Total rows    : {summary[0]:,}")
print(f"   Date range    : {summary[1]} -> {summary[2]}")
print(f"   Unique stores : {summary[3]}")
print(f"   Unique SKUs   : {summary[4]:,}")
print(f"   Total columns : 28")
print("=== ALL PIPELINE STEPS AND VALIDATION ASSERTIONS COMPLETED SUCCESSFULLY! ===")